# Week 1 Comparison Notebook

This notebook combines the Benford result with the two supervised models.
It saves the main comparison table to `data/generated/model_comparison.csv`.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


In [2]:
from pathlib import Path

# This makes the notebook work whether you open it from the repo root
# or from inside the notebooks folder.
if (Path.cwd() / "data").exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    REPO_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not find the repo root")

DATA_DIR = REPO_ROOT / "data" / "training" / "vynfi"
OUT_DIR = REPO_ROOT / "data" / "generated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Data folder:", DATA_DIR)


Repo root: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint
Data folder: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint\data\training\vynfi


In [3]:
# I load the same three files here so this comparison can run on its own.
shard_names = [
    "train-00000-of-00003.parquet",
    "train-00001-of-00003.parquet",
    "train-00002-of-00003.parquet",
]
main_data = pd.concat([pd.read_parquet(DATA_DIR / name) for name in shard_names], ignore_index=True)

empty_columns = [
    "auxiliary_account_number",
    "auxiliary_account_label",
    "lettrage",
    "lettrage_date",
    "tax_code",
]
leakage_columns = ["fraud_type", "anomaly_type", "is_anomaly"]

work_df = main_data.drop(columns=empty_columns, errors="ignore")

# I use the same document-level split for every method to make the comparison fair.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_index, test_index = next(
    splitter.split(work_df, y=work_df["is_fraud"], groups=work_df["document_id"])
)

train_df = work_df.iloc[train_index].reset_index(drop=True)
test_df = work_df.iloc[test_index].reset_index(drop=True)
eval_fraud_type = test_df["fraud_type"].copy()

y_train = train_df["is_fraud"].astype(int)
y_test = test_df["is_fraud"].astype(int)

train_df = train_df.drop(columns=leakage_columns, errors="ignore")
test_df = test_df.drop(columns=leakage_columns, errors="ignore")


In [4]:
# This function puts every method's results into the same columns.
def score_row(method_name, y_true, y_pred, y_score=None, notes=""):
    row = {
        "method": method_name,
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "average_precision": np.nan,
        "roc_auc": np.nan,
        "n_flagged": int(np.sum(y_pred)),
        "notes": notes,
    }
    if y_score is not None:
        row["average_precision"] = round(average_precision_score(y_true, y_score), 4)
        row["roc_auc"] = round(roc_auc_score(y_true, y_score), 4)
    return row


In [5]:
# I first run Benford as a rule-based baseline that does not learn from fraud labels.
expected_digits = np.arange(1, 10)
expected_shares = np.log10(1 + 1 / expected_digits)
# Skip amounts below one cent, use the standard 0.015 MAD limit, and require 500 rows to reduce random variation.

def line_amounts(df):
    return (df["debit_amount"].fillna(0) + df["credit_amount"].fillna(0)).abs()


def first_digits(amount_series):
    usable = amount_series[amount_series >= 0.01]
    scaled = usable / np.power(10.0, np.floor(np.log10(usable)))
    return scaled.astype(int).clip(1, 9)


def digit_shares(digit_series):
    counts = digit_series.value_counts().reindex(expected_digits, fill_value=0).to_numpy(dtype=float)
    total = counts.sum()
    if total == 0:
        return np.zeros(9)
    return counts / total


def mad_score(observed):
    return float(np.mean(np.abs(observed - expected_shares)))

# Accounts are selected from training data only, so the test result stays separate.
train_digits = first_digits(line_amounts(train_df))
train_benford = pd.DataFrame(
    {
        "gl_account": train_df.loc[train_digits.index, "gl_account"].to_numpy(),
        "digit": train_digits.to_numpy(),
    }
)

benford_rows = []
for gl_account, chunk in train_benford.groupby("gl_account"):
    observed = digit_shares(chunk["digit"])
    mad_value = mad_score(observed)
    benford_rows.append(
        {
            "gl_account": gl_account,
            "rows": len(chunk),
            "mad": mad_value,
            "testable": len(chunk) >= 500,
            "flagged": len(chunk) >= 500 and mad_value > 0.015,
        }
    )

benford_groups = pd.DataFrame(benford_rows)
failed_accounts = set(benford_groups.loc[benford_groups["flagged"], "gl_account"])
benford_pred = test_df["gl_account"].isin(failed_accounts).astype(int)


In [6]:
# I then train supervised models that can learn patterns from the fraud labels.
numeric_features = [
    "amount",
    "log10_amount",
    "first_digit",
    "posting_lag_days",
    "posting_dayofweek",
    "exchange_rate",
    "line_number",
    "fiscal_period",
]
boolean_features = [
    "is_debit",
    "is_round_100",
    "is_round_1000",
    "is_weekend",
    "is_manual",
    "is_post_close",
]
categorical_features = [
    "document_type",
    "currency",
    "business_process",
    "business_unit",
    "account_class",
    "account_sub_class",
    "financial_statement_category",
    "source_system",
    "company_code",
]


# These features describe the amount, timing and accounting details of each row.
def build_features(df):
    out = pd.DataFrame(index=df.index)
    debit = df["debit_amount"].fillna(0)
    credit = df["credit_amount"].fillna(0)
    amount = (debit + credit).abs()
    out["amount"] = amount
    out["log10_amount"] = np.log10(amount.clip(lower=0.01))
    scaled = amount.clip(lower=0.01) / np.power(10.0, np.floor(np.log10(amount.clip(lower=0.01))))
    out["first_digit"] = scaled.astype(int).clip(1, 9)
    out["is_debit"] = (debit > 0).astype(int)
    out["is_round_100"] = ((amount > 0) & (amount % 100 == 0)).astype(int)
    out["is_round_1000"] = ((amount > 0) & (amount % 1000 == 0)).astype(int)
    lag = (df["posting_date"] - df["document_date"]).dt.days
    out["posting_lag_days"] = lag.fillna(0)
    out["posting_dayofweek"] = df["posting_date"].dt.dayofweek.fillna(0)
    out["is_weekend"] = (df["posting_date"].dt.dayofweek >= 5).astype(int)
    out["exchange_rate"] = df["exchange_rate"].fillna(1.0)
    out["line_number"] = df["line_number"].fillna(0)
    out["fiscal_period"] = df["fiscal_period"].fillna(0)
    out["is_manual"] = df["is_manual"].astype(int)
    out["is_post_close"] = df["is_post_close"].astype(int)
    for column in categorical_features:
        out[column] = df[column].astype("string").fillna("missing")
    out["gl_account"] = df["gl_account"].fillna(-1).astype("int64")
    return out

X_train = build_features(train_df)
X_test = build_features(test_df)

# Logistic regression uses scaled numbers and one-hot encoded categories.
linear_preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
            numeric_features + boolean_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=True),
            categorical_features + ["gl_account"],
        ),
    ],
    remainder="drop",
)

logistic_model = Pipeline(
    [
        ("prep", linear_preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, n_jobs=-1)),
    ]
)
# Balanced class weights help because normal rows are much more common than fraud rows.
logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_score = logistic_model.predict_proba(X_test)[:, 1]

n_numeric = len(numeric_features) + len(boolean_features) + 1
categorical_positions = list(range(n_numeric, n_numeric + len(categorical_features)))

# Gradient boosting can learn non-linear patterns, so I use it as the second model.
tree_preprocessor = ColumnTransformer(
    [
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features + boolean_features + ["gl_account"],
        ),
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1),
            categorical_features,
        ),
    ],
    remainder="drop",
)

gradient_model = Pipeline(
    [
        ("prep", tree_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.1,
                class_weight="balanced",
                categorical_features=categorical_positions,
                random_state=42,
            ),
        ),
    ]
)
gradient_model.fit(X_train, y_train)
gradient_pred = gradient_model.predict(X_test)
gradient_score = gradient_model.predict_proba(X_test)[:, 1]


C:\Users\svgow\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\svgow\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\svgow\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\svgow\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\svgow\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

In [7]:
# One final table makes the strengths and weaknesses of all three methods visible.
results_table = pd.DataFrame(
    [
        score_row("Benford (MAD by gl_account)", y_test, benford_pred, notes="unsupervised, group level, no labels used"),
        score_row("Logistic regression", y_test, logistic_pred, logistic_score, notes="supervised, class_weight balanced"),
        score_row("Gradient boosting", y_test, gradient_pred, gradient_score, notes="supervised, class_weight balanced"),
    ]
)

results_table.to_csv(OUT_DIR / "model_comparison.csv", index=False)
results_table


,method,precision,recall,f1,average_precision,roc_auc,n_flagged,notes
0,Benford (MAD by gl_account),0.0576,0.0924,0.0710,NaN,NaN,13035,"unsupervised, group level, no labels used"
1,Logistic regression,0.3941,0.6542,0.4919,0.6724,0.8425,13491,"supervised, class_weight balanced"
2,Gradient boosting,0.7230,0.6675,0.6942,0.7176,0.8472,7503,"supervised, class_weight balanced"
